# Datathon Task 2 V3 — OCR Recovery and Route Reverse Engineering

**Objective:** memperbaiki V2 berdasarkan error OOF yang benar-benar terukur, bukan sekadar menambah kompleksitas model.

Diagnosis V2 membagi 2.166 error menjadi 1.723 shortest-path tie-break errors, 295 cases ketika true link bukan shortest pada graph OCR, dan 148 candidate misses. Raw OCR memperlihatkan merged regions seperti `Scotland Philosophy` atau `Hurricane Katrina 2005 Atlantic hurricane season` yang sebenarnya berisi beberapa link tetapi V1 hanya menerima satu candidate.

V3 melakukan:

1. exact multi-title segmentation dan conservative fuzzy recovery pada suspicious OCR regions;
2. rebuild hyperlink graph dan reverse-BFS distances;
3. grouped OOF comparison terhadap V1/V2;
4. hard shortest-path tie-break search pada graph yang sudah dipulihkan;
5. explicit error taxonomy untuk menentukan ceiling dan arah V4.

Local full run adalah default dan seluruh output disimpan ke `patrick/output/v3/`.

## 1. Setup dan reproducibility

Cell ini mengatur seed, input, output, dan feature-search budget. V2 sengaja tidak memanggil Tesseract/OpenCV. Local full run adalah default dan memakai path absolut project Patrick; set `DATATON_LOCAL_SMOKE=1` hanya jika ingin subset cepat. Kaggle path dan auto-discovery tetap tersedia sebagai fallback portable.

In [ ]:
from __future__ import annotations

import gc, json, math, os, shutil, time, warnings
from collections import Counter, defaultdict, deque
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

IS_KAGGLE = Path('/kaggle/working').exists()
LOCAL_SMOKE_MODE = (not IS_KAGGLE) and os.getenv('DATATON_LOCAL_SMOKE','0') == '1'
N_SPLITS = 3
MAX_CANDIDATES = 100
N_SOFT_SEARCH = 24 if LOCAL_SMOKE_MODE else 400
N_HARDMIN_SEARCH = 24 if LOCAL_SMOKE_MODE else 400
RUN_LAMBDARANK = not LOCAL_SMOKE_MODE
LOCAL_SMOKE_TARGETS = 30
START_TIME = time.time()

KAGGLE_COMPETITION_PATH = Path('/kaggle/input/competitions/datathon-task-2/dataset-task2')
KAGGLE_REUSE_PATH = Path('/kaggle/input/datasets/timotiuspatrick/datatonday2output/datathon_task2_v1_reusable')
KAGGLE_REUSE_SHORT_PATH = Path('/kaggle/input/datatonday2output/datathon_task2_v1_reusable')
LOCAL_PATRICK_PATH = Path('/Users/timotiuspatrick/Documents/Patrick/Kuliah & Coding/Dataton/day 2/patrick')
LOCAL_DATA_PATH = LOCAL_PATRICK_PATH.parent/'dataset-task2'
LOCAL_REUSE_PATH = LOCAL_PATRICK_PATH/'output/v1/datathon_task2_v1_reusable'
LOCAL_OUTPUT_PATH = LOCAL_PATRICK_PATH/'output/v3'

DATA_PATH = KAGGLE_COMPETITION_PATH if KAGGLE_COMPETITION_PATH.exists() else LOCAL_DATA_PATH
OUTPUT_DIR = Path('/kaggle/working/datathon_task2_v3') if IS_KAGGLE else LOCAL_OUTPUT_PATH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def valid_reuse(path):
    return (path/'manifest.json').exists() and (path/'task2_v1_link_candidates.csv').exists()

reuse_candidates = [KAGGLE_REUSE_PATH, KAGGLE_REUSE_SHORT_PATH, LOCAL_REUSE_PATH] if IS_KAGGLE else [LOCAL_REUSE_PATH, KAGGLE_REUSE_PATH, KAGGLE_REUSE_SHORT_PATH]
REUSE_PATH = next((p for p in reuse_candidates if valid_reuse(p)), None)
if REUSE_PATH is None and Path('/kaggle/input').exists():
    discovered = [p.parent for p in Path('/kaggle/input').rglob('manifest.json') if valid_reuse(p.parent)]
    REUSE_PATH = discovered[0] if discovered else None
if REUSE_PATH is None:
    raise FileNotFoundError('Reusable V1 folder tidak ditemukan. Attach Kaggle Dataset datatonday2output terlebih dahulu.')

print('Competition data :', DATA_PATH)
print('Reusable V1 data :', REUSE_PATH)
print('Output directory :', OUTPUT_DIR)
execution_label = 'LOCAL SMOKE/SUBSET' if LOCAL_SMOKE_MODE else ('KAGGLE FULL' if IS_KAGGLE else 'LOCAL FULL')
print('Execution mode   :', execution_label)
print('Search budgets   :', N_SOFT_SEARCH, '+', N_HARDMIN_SEARCH)
print('LightGBM enabled :', RUN_LAMBDARANK)

## 2. Load dan validasi kontrak input

Manifest harus menyatakan seluruh 4.604 halaman sudah diproses. Cell ini juga memastikan tabel kompetisi dan reusable graph memiliki schema yang dibutuhkan. Jika kontrak gagal, notebook berhenti sebelum modeling agar tidak diam-diam memakai cache parsial.

In [ ]:
manifest = json.loads((REUSE_PATH/'manifest.json').read_text())
assert manifest['is_complete'] is True
assert manifest['processed_pages'] == manifest['expected_pages'] == 4604

articles = pd.read_csv(DATA_PATH/'articles.csv').sort_values('article_id').reset_index(drop=True)
categories = pd.read_csv(DATA_PATH/'categories.csv')
train = pd.read_csv(DATA_PATH/'states_train.csv')
test = pd.read_csv(DATA_PATH/'states_test.csv')
sample = pd.read_csv(DATA_PATH/'sample_submission.csv')
links = pd.read_csv(REUSE_PATH/'task2_v1_link_candidates.csv')
raw_ocr = pd.read_csv(REUSE_PATH/'task2_v1_ocr_raw_regions.csv')
quality = pd.read_csv(REUSE_PATH/'task2_v1_ocr_quality_by_page.csv')

assert list(train.columns) == ['state_id','current_article_id','target_article_id','next_article_id']
assert list(test.columns) == ['state_id','current_article_id','target_article_id']
assert sample.state_id.tolist() == test.state_id.tolist()
assert links[['source_id','candidate_id']].notna().all().all()
assert articles.article_id.tolist() == list(range(4604))

input_summary = pd.DataFrame({
    'artifact':['articles','train','test','accepted graph edges','raw OCR regions','processed pages'],
    'rows':[len(articles),len(train),len(test),len(links),len(raw_ocr),manifest['processed_pages']]
})
display(input_summary)
if LOCAL_SMOKE_MODE:
    smoke_targets=sorted(train.target_article_id.unique())[:LOCAL_SMOKE_TARGETS]
    dev_train=train[train.target_article_id.isin(smoke_targets)].copy()
    print(f'LOCAL SMOKE: validation/search memakai {len(dev_train):,} rows dan {len(smoke_targets)} targets; full run wajib di Kaggle.')
else:
    dev_train=train.copy()
print('Reusable bundle complete:', manifest['is_complete'])

## 3. Ringkasan V1 dan gap yang ingin ditutup

V1 sudah menemukan kandidat dengan baik, tetapi top-1 masih jauh dari top-3/top-5. Visual berikut memisahkan extraction ceiling dari ranking gap. Angka V1 dibaca dari diagnostics jika file tersebut ikut terpasang; fallback memakai hasil full run yang sudah diaudit.

In [ ]:
diag_candidates = list(REUSE_PATH.parent.glob('datathon_task2_v1_diagnostics.json'))
if diag_candidates:
    v1_diag = json.loads(diag_candidates[0].read_text())
    v1_cv = pd.DataFrame(v1_diag['cv_summary']).set_index('model')
    v1_accuracy = float(v1_cv.loc['heuristic','mean_accuracy'])
    v1_top3 = float(v1_cv.loc['heuristic','mean_top3'])
    v1_top5 = float(v1_cv.loc['heuristic','mean_top5'])
    v1_recall = float(v1_diag['graph_eda']['raw_ocr_candidate_recall'])
else:
    v1_accuracy, v1_top3, v1_top5, v1_recall = 0.6508888889, 0.8991111111, 0.948, 0.9783333333

v1_gap = pd.Series({
    'V1 top-1':v1_accuracy,
    'V1 top-3':v1_top3,
    'V1 top-5':v1_top5,
    'Raw candidate recall':v1_recall,
})
display(v1_gap.to_frame('score'))
fig, ax = plt.subplots(figsize=(9,4))
v1_gap.plot.bar(ax=ax, color=['#5965a8','#4c9f70','#d99b3d','#9b59b6'])
ax.set_ylim(0,1); ax.set_ylabel('accuracy / recall'); ax.set_title('V1: extraction kuat, bottleneck ada pada top-1 ranking')
ax.tick_params(axis='x', rotation=15); plt.tight_layout(); plt.show()
print(f'INSIGHT V1: gap top-1 ke top-3 = {v1_top3-v1_accuracy:.2%}; V2 harus memperbaiki tie-breaking, bukan mengulang OCR.')

## 4. OCR merged-region recovery

V1 mempunyai accepted rate tinggi, tetapi connected components kadang menggabungkan dua atau lebih blue links menjadi satu OCR string. V3 memindai suspicious regions secara unsupervised:

- exact full-title segmentation dengan word boundaries;
- conservative fuzzy match dengan cutoff 90;
- OCR confusion recovery seperti `lron -> Iron` hanya dengan cutoff lebih ketat;
- self-links dan duplicate edges dibuang.

Tidak ada label train yang dipakai untuk menentukan recovered edge, sehingga graph repair aman untuk validation dan test.

In [ ]:

import re
from rapidfuzz import fuzz, process

def normalize_ocr_text(value):
    return re.sub(r'[^a-z0-9]+',' ',str(value).casefold()).strip()

title_map=articles.set_index('article_id').title.to_dict()
normalized_titles={aid:normalize_ocr_text(title) for aid,title in title_map.items()}
title_choices={aid:value for aid,value in normalized_titles.items() if len(value)>=4}
candidate_title_len=raw_ocr.candidate_title.fillna('').astype(str).str.len()
ocr_text_len=raw_ocr.ocr_text.fillna('').astype(str).str.len()
suspect_regions=raw_ocr[(raw_ocr.status.ne('accepted'))|(ocr_text_len>candidate_title_len+3)].copy()

recovery_rows=[]
for row in suspect_regions.itertuples(index=False):
    text=normalize_ocr_text(row.ocr_text)
    if not text: continue
    padded=f' {text} '; found=set()
    for article_id,title_norm in title_choices.items():
        if f' {title_norm} ' in padded: found.add(int(article_id))
    method='exact_segment'
    if not found:
        method='fuzzy'
        for _,score,article_id in process.extract(text,title_choices,scorer=fuzz.WRatio,limit=3,score_cutoff=90):
            found.add(int(article_id))
        if text.startswith('l'):
            variant='i'+text[1:]
            for _,score,article_id in process.extract(variant,title_choices,scorer=fuzz.WRatio,limit=2,score_cutoff=94):
                found.add(int(article_id))
    for article_id in found:
        if article_id==int(row.source_id): continue
        actual_method='exact_segment' if f' {normalized_titles[article_id]} ' in padded else method
        recovery_rows.append({'source_id':int(row.source_id),'candidate_id':article_id,'ocr_text':str(row.ocr_text),'ocr_confidence':float(row.ocr_confidence),'match_score':1.0,'position_y':float(row.position_y),'box_x':float(row.box_x),'recovery_method':actual_method})

recovery=pd.DataFrame(recovery_rows).drop_duplicates(['source_id','candidate_id'])
existing_edges=set(map(tuple,links[['source_id','candidate_id']].astype(int).to_numpy()))
recovery=recovery[[tuple(x) not in existing_edges for x in recovery[['source_id','candidate_id']].to_numpy()]].reset_index(drop=True)
links_v3=pd.concat([links,recovery[links.columns]],ignore_index=True).sort_values(['source_id','candidate_id','match_score'],ascending=[True,True,False]).drop_duplicates(['source_id','candidate_id'])

recovery_summary=pd.Series({'suspect_regions':len(suspect_regions),'new_recovered_edges':len(recovery),'exact_segment':int(recovery.recovery_method.eq('exact_segment').sum()),'fuzzy':int(recovery.recovery_method.eq('fuzzy').sum()),'v1_edges':len(links),'v3_edges':len(links_v3)})
display(recovery_summary.to_frame('value'))
fig,ax=plt.subplots(1,2,figsize=(12,4)); recovery.recovery_method.value_counts().plot.bar(ax=ax[0],color=['#4c78a8','#f58518']); ax[0].set_title('Recovered edges by method'); sns.histplot(recovery.position_y,bins=30,ax=ax[1]); ax[1].set_title('Recovered-link vertical position'); plt.tight_layout(); plt.show()
display(recovery[['source_id','candidate_id','ocr_text','recovery_method']].head(20))
print(f'INSIGHT RECOVERY: {len(recovery):,} new edges recovered without using train labels.')


## 4. Rekonstruksi graph dan metadata link

Accepted OCR links diubah menjadi directed adjacency graph. Raw OCR yang memiliki kandidat valid diagregasi agar kandidat yang muncul beberapa kali dapat memiliki `mention_count`, confidence, dan posisi pertama. Feature graph/category dibuat sekali dan digunakan ulang seluruh fold.

In [ ]:
n_articles = int(articles.article_id.max()) + 1
titles = articles.set_index('article_id').title.to_dict()
ocr_candidates = defaultdict(set)
for r in links_v3.itertuples(index=False):
    ocr_candidates[int(r.source_id)].add(int(r.candidate_id))

accepted_raw = raw_ocr[raw_ocr.status.eq('accepted') & raw_ocr.candidate_id.notna()].copy()
accepted_raw['candidate_id'] = accepted_raw.candidate_id.astype(int)
raw_agg = accepted_raw.groupby(['source_id','candidate_id']).agg(
    ocr_conf_max=('ocr_confidence','max'), ocr_conf_mean=('ocr_confidence','mean'),
    title_match_max=('match_score','max'), first_position=('position_y','min'),
    mention_count=('candidate_id','size')
).reset_index()
raw_meta = {
    (int(r.source_id),int(r.candidate_id)):(float(r.ocr_conf_max),float(r.ocr_conf_mean),float(r.title_match_max),float(r.first_position),int(r.mention_count))
    for r in raw_agg.itertuples(index=False)
}

for r in recovery.itertuples(index=False):
    raw_meta[(int(r.source_id),int(r.candidate_id))]=(float(r.ocr_confidence),float(r.ocr_confidence),1.0,float(r.position_y),1)

out_degree = np.zeros(n_articles,dtype=np.int32)
in_degree = np.zeros(n_articles,dtype=np.int32)
reverse_adj = [[] for _ in range(n_articles)]
incoming_sets = [set() for _ in range(n_articles)]
for source,candidates in ocr_candidates.items():
    out_degree[source] = len(candidates)
    for candidate in candidates:
        in_degree[candidate] += 1
        reverse_adj[candidate].append(source)
        incoming_sets[candidate].add(source)

graph_summary = pd.Series({
    'directed_edges':int(out_degree.sum()),
    'pages_with_links':int((out_degree>0).sum()),
    'pages_without_links':int((out_degree==0).sum()),
    'median_out_degree':float(np.median(out_degree[out_degree>0])),
    'p95_out_degree':float(np.percentile(out_degree[out_degree>0],95)),
})
display(graph_summary.to_frame('value'))

## 5. Visual EDA graph dan OCR cache

Distribusi degree menunjukkan ukuran candidate set sebenarnya. Plot quality tidak dianggap sebagai precision ground-truth; accepted rate hanya mendiagnosis perilaku matcher. Train candidate recall tetap menjadi ukuran extraction yang lebih relevan.

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(17,4))
sns.histplot(out_degree[out_degree>0],bins=40,ax=ax[0],color='#4c78a8'); ax[0].set_title('Outgoing candidates per page')
sns.histplot(in_degree[in_degree>0],bins=40,ax=ax[1],color='#f58518',log_scale=(False,True)); ax[1].set_title('Incoming degree (log count axis)')
sns.histplot(quality.accepted_rate,bins=30,ax=ax[2],color='#54a24b'); ax[2].set_title('OCR matcher accepted rate')
plt.tight_layout(); plt.show()

low_quality = quality.sort_values(['accepted_rate','regions']).head(12)
display(low_quality)
print('INSIGHT GRAPH: candidate cap 100 mempertahankan hampir seluruh page, sedangkan long-tail degree mencegah pair table membesar tanpa batas.')

## 6. Representasi semantic dan category hierarchy

V2 tetap menyimpan word/character TF-IDF sebagai tie-break ringan, tetapi menambah category hierarchy, exact overlap count, shared outgoing neighbours, dan shared incoming neighbours. Audit lokal menunjukkan exact category jauh lebih penting daripada similarity judul pada shortest-path ties.

In [ ]:
category_lists = categories.groupby('article_id').category.apply(list).to_dict()
category_sets = {i:set(category_lists.get(i,[])) for i in range(n_articles)}
category_tokens = {
    i:set(token.lower() for cat in category_lists.get(i,[]) for token in str(cat).replace('subject.','').replace('.',' ').replace('_',' ').split())
    for i in range(n_articles)
}
category_prefixes = {}
for aid in range(n_articles):
    prefixes=set()
    for category in category_lists.get(aid,[]):
        parts=str(category).split('.')
        prefixes.update('.'.join(parts[:depth]) for depth in range(2,len(parts)+1))
    category_prefixes[aid]=prefixes

docs=[]
for aid in range(n_articles):
    cat_doc=' '.join(str(x).replace('subject.',' ').replace('.',' ').replace('_',' ') for x in category_lists.get(aid,[]))
    docs.append(f'{titles[aid]} {titles[aid]} {cat_doc}')
word_x=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True).fit_transform(docs)
char_x=TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=2,sublinear_tf=True,max_features=30000).fit_transform(docs)
print('Word TF-IDF:',word_x.shape,'| Char TF-IDF:',char_x.shape)

## 7. Reverse BFS dan leakage-safe prior

Reverse BFS menghitung jarak setiap kandidat menuju target pada graph hasil screenshot. Cache dibuat per target sehingga 600 target hanya dihitung sekali. Click-prior training memakai leave-one-row-out counts: label sebuah row tidak boleh menciptakan feature prior untuk row itu sendiri.

In [ ]:
dist_cache={}
def distances_to_target(target):
    target=int(target)
    if target in dist_cache: return dist_cache[target]
    dist=np.full(n_articles,99,dtype=np.int16); dist[target]=0
    queue=deque([target])
    while queue:
        node=queue.popleft(); nd=int(dist[node])+1
        for previous in reverse_adj[node]:
            if dist[previous]==99:
                dist[previous]=nd; queue.append(previous)
    dist_cache[target]=dist
    return dist

def sparse_cosine(matrix,candidates,target):
    return np.asarray(matrix[candidates].multiply(matrix[int(target)]).sum(axis=1)).ravel().astype(np.float32)

def jaccard(a,b):
    return len(a&b)/len(a|b) if (a or b) else 0.0

def fit_priors(frame):
    global_counts=Counter(frame.next_article_id.astype(int))
    current_counts=defaultdict(Counter)
    for r in frame.itertuples(index=False): current_counts[int(r.current_article_id)][int(r.next_article_id)]+=1
    return global_counts,current_counts,[x for x,_ in global_counts.most_common(30)]

## 8. Candidate-pair features

Setiap state menjadi satu ranking query. Kandidat berasal dari OCR graph ditambah transition yang diamati hanya pada fold-training. Positive label boleh ditambahkan pada training pair agar ranker memiliki positive example, tetapi prior count-nya dikurangi satu untuk menghapus self-label leakage.

In [ ]:
FEATURES=[
 'direct_target','word_cosine','char_cosine','category_exact_jaccard','category_token_jaccard',
 'graph_distance','graph_improvement','on_shortest_path','is_min_neighbor_distance','distance_tie_count',
 'ocr_conf_max','ocr_conf_mean','title_match_max','first_position','log_mentions','log_global_clicks',
 'log_current_clicks','observed_edge','log_out_degree','log_in_degree','word_rank','char_rank','position_rank',
 'distance_rank','category_overlap_count','category_prefix_jaccard','out_neighbor_jaccard',
 'in_neighbor_jaccard','reciprocal_link'
]

def build_pairs(frame,priors,training=False):
    global_counts,current_counts,global_top=priors
    blocks=[]; labels=[]; states=[]; candidates_out=[]
    has_label='next_article_id' in frame.columns
    for row in frame.itertuples(index=False):
        state,current,target=int(row.state_id),int(row.current_article_id),int(row.target_article_id)
        truth=int(row.next_article_id) if has_label else None
        observed=set(current_counts.get(current,{}))
        candidates=set(ocr_candidates.get(current,set()))|observed
        if not candidates: candidates=set(global_top[:20])|{target}
        if training and truth is not None: candidates.add(truth)
        def priority(candidate):
            conf,_,match,position,mentions=raw_meta.get((current,candidate),(0,0,0,1,0))
            return candidate==truth,candidate==target,candidate in observed,match,conf,mentions,-position
        candidate_list=sorted(candidates,key=priority,reverse=True)[:MAX_CANDIDATES]
        cands=np.asarray(candidate_list,dtype=int)
        dist=distances_to_target(target); cdist=np.minimum(dist[cands].astype(np.float32),20)
        current_dist=min(int(dist[current]),20); min_dist=float(cdist.min()); tie_count=int((cdist==min_dist).sum())
        word=sparse_cosine(word_x,cands,target); char=sparse_cosine(char_x,cands,target)
        positions=np.asarray([raw_meta.get((current,int(c)),(0,0,0,1,0))[3] for c in cands],dtype=np.float32)
        block=np.zeros((len(cands),len(FEATURES)),dtype=np.float32)
        for idx,candidate in enumerate(cands):
            candidate=int(candidate); conf_max,conf_mean,match,position,mentions=raw_meta.get((current,candidate),(0,0,0,1,0))
            global_count=global_counts.get(candidate,0); current_count=current_counts.get(current,{}).get(candidate,0)
            if training and truth is not None and candidate==truth:
                global_count=max(0,global_count-1); current_count=max(0,current_count-1)
            block[idx,:20]=[
                candidate==target,word[idx],char[idx],jaccard(category_sets[candidate],category_sets[target]),
                jaccard(category_tokens[candidate],category_tokens[target]),cdist[idx],current_dist-cdist[idx],
                int(current_dist<20 and cdist[idx]+1==current_dist),cdist[idx]==min_dist,tie_count,
                conf_max,conf_mean,match,position,math.log1p(mentions),math.log1p(global_count),
                math.log1p(current_count),current_count>0,math.log1p(out_degree[candidate]),math.log1p(in_degree[candidate])]
            block[idx,24:]=[
                len(category_sets[candidate]&category_sets[target]),jaccard(category_prefixes[candidate],category_prefixes[target]),
                jaccard(ocr_candidates.get(candidate,set()),ocr_candidates.get(target,set())),
                jaccard(incoming_sets[candidate],incoming_sets[target]),candidate in ocr_candidates.get(target,set())]
        block[:,20]=pd.Series(word).rank(method='average',pct=True).to_numpy(np.float32)
        block[:,21]=pd.Series(char).rank(method='average',pct=True).to_numpy(np.float32)
        block[:,22]=pd.Series(-positions).rank(method='average',pct=True).to_numpy(np.float32)
        block[:,23]=pd.Series(-cdist).rank(method='average',pct=True).to_numpy(np.float32)
        blocks.append(block); labels.extend((cands==truth).astype(np.int8) if truth is not None else np.zeros(len(cands),dtype=np.int8))
        states.extend([state]*len(cands)); candidates_out.extend(cands.tolist())
    return np.vstack(blocks),np.asarray(labels,np.int8),np.asarray(states,np.int32),np.asarray(candidates_out,np.int32)

def evaluate_scores(y,states,candidates,scores):
    order=np.lexsort((candidates,-scores,states)); sorted_states=states[order]
    first=np.r_[True,sorted_states[1:]!=sorted_states[:-1]]; winners=order[first]
    ranked=pd.DataFrame({'state':states,'candidate':candidates,'truth':y,'score':scores}).sort_values(['state','score','candidate'],ascending=[True,False,True])
    ranked['rank']=ranked.groupby('state').cumcount()+1
    n_states=len(np.unique(states))
    return {
        'accuracy':float(y[winners].mean()),
        'top3':float(((ranked.truth==1)&(ranked['rank']<=3)).groupby(ranked.state).max().sum()/n_states),
        'top5':float(((ranked.truth==1)&(ranked['rank']<=5)).groupby(ranked.state).max().sum()/n_states),
        'candidate_recall':float(y.sum()/n_states),
    }

def winner_table(y,states,candidates,scores):
    order=np.lexsort((candidates,-scores,states)); sorted_states=states[order]
    first=np.r_[True,sorted_states[1:]!=sorted_states[:-1]]; idx=order[first]
    return pd.DataFrame({'state_id':states[idx],'prediction':candidates[idx],'correct':y[idx].astype(bool),'score':scores[idx]})

## 9. Grouped fold feature cache dan route diagnostics

Seluruh 25 states untuk satu target tetap berada di fold yang sama. Feature cache dibangun sekali agar berbagai scoring rule membandingkan candidate set identik. Diagnostics mengukur berapa sering truth tersedia, berada pada minimum distance, dan menjadi satu-satunya minimum-distance candidate.

In [ ]:
fold_cache=[]; route_rows=[]
splitter=GroupKFold(n_splits=N_SPLITS)
for fold,(fit_idx,val_idx) in enumerate(splitter.split(dev_train,groups=dev_train.target_article_id)):
    fit=dev_train.iloc[fit_idx].copy(); val=dev_train.iloc[val_idx].copy(); priors=fit_priors(fit)
    x_fit,y_fit,q_fit,c_fit=build_pairs(fit,priors,training=True)
    x_val,y_val,q_val,c_val=build_pairs(val,priors,training=False)
    state_count=len(np.unique(q_val)); positive=y_val==1
    tie_idx=FEATURES.index('distance_tie_count'); min_idx=FEATURES.index('is_min_neighbor_distance')
    state_ties=pd.DataFrame({'state':q_val,'ties':x_val[:,tie_idx]}).drop_duplicates('state')
    route_rows.append({
        'fold':fold,'candidate_recall':positive.sum()/state_count,
        'truth_is_min_distance':(positive&(x_val[:,min_idx]==1)).sum()/state_count,
        'truth_is_unique_min':(positive&(x_val[:,min_idx]==1)&(x_val[:,tie_idx]==1)).sum()/state_count,
        'states_unique_min':state_ties.ties.eq(1).mean(),
    })
    fold_cache.append({'fold':fold,'fit':(x_fit,y_fit,q_fit,c_fit),'val':(x_val,y_val,q_val,c_val)})
    print(f'Fold {fold}: train pairs={len(x_fit):,} | val pairs={len(x_val):,}')

route_df=pd.DataFrame(route_rows)
display(route_df); display(route_df.mean(numeric_only=True).to_frame('mean'))
fig,ax=plt.subplots(1,2,figsize=(13,4))
route_df.set_index('fold')[['candidate_recall','truth_is_min_distance','truth_is_unique_min']].plot.bar(ax=ax[0]); ax[0].set_ylim(0,1); ax[0].set_title('Route diagnostics by fold')
all_ties=pd.concat([pd.DataFrame({'state':f['val'][2],'ties':f['val'][0][:,FEATURES.index('distance_tie_count')]}) for f in fold_cache]).drop_duplicates('state')
sns.histplot(all_ties.ties.clip(upper=15),bins=15,discrete=True,ax=ax[1]); ax[1].set_title('Number of minimum-distance candidates (15=15+)')
plt.tight_layout(); plt.show()
print(f"INSIGHT ROUTE: {route_df.truth_is_min_distance.mean():.2%} truth berada di minimum distance, tetapi hanya {route_df.states_unique_min.mean():.2%} state memiliki minimum yang unik.")

## 10. Seeded soft dan hard-min heuristic search

Soft search masih mengizinkan kandidat non-minimum menang. Hard-min search memberi margin 100 kepada kandidat minimum-distance, lalu mengoptimalkan tie-break berbasis category, popularity, graph neighbourhood, semantic similarity, dan posisi. Seluruh konfigurasi identik pada ketiga fold dan seed tersimpan agar hasil reproducible.

In [ ]:
SOFT_FEATURES=['direct_target','graph_distance','is_min_neighbor_distance','char_cosine','word_cosine','category_exact_jaccard','category_token_jaccard','log_global_clicks','log_current_clicks','position_rank','log_in_degree','category_overlap_count','category_prefix_jaccard','out_neighbor_jaccard','in_neighbor_jaccard','reciprocal_link']
TIE_FEATURES=['is_min_neighbor_distance','category_exact_jaccard','category_overlap_count','category_prefix_jaccard','log_global_clicks','log_in_degree','out_neighbor_jaccard','in_neighbor_jaccard','char_cosine','word_cosine','position_rank']
rng=np.random.default_rng(SEED)
soft_weights=[]
for sid in range(N_SOFT_SEARCH):
    soft_weights.append({'search_id':sid,'direct_target':20.0,'graph_distance':-float(rng.uniform(.15,1.4)),'is_min_neighbor_distance':float(rng.uniform(0,2)),'char_cosine':float(rng.uniform(0,3)),'word_cosine':float(rng.uniform(0,2)),'category_exact_jaccard':float(rng.uniform(0,1.5)),'category_token_jaccard':float(rng.uniform(0,1.5)),'log_global_clicks':float(rng.uniform(0,.25)),'log_current_clicks':float(rng.uniform(0,.35)),'position_rank':float(rng.uniform(0,.5)),'log_in_degree':float(rng.uniform(-.1,.2)),'category_overlap_count':float(rng.uniform(0,1)),'category_prefix_jaccard':float(rng.uniform(0,1.5)),'out_neighbor_jaccard':float(rng.uniform(0,1)),'in_neighbor_jaccard':float(rng.uniform(0,1)),'reciprocal_link':float(rng.uniform(-.2,.5))})
tie_weights=[]
for sid in range(N_HARDMIN_SEARCH):
    tie_weights.append({'search_id':sid,'is_min_neighbor_distance':100.0,'category_exact_jaccard':float(rng.uniform(0,3)),'category_overlap_count':float(rng.uniform(0,1.5)),'category_prefix_jaccard':float(rng.uniform(0,2.5)),'log_global_clicks':float(rng.uniform(0,.6)),'log_in_degree':float(rng.uniform(-.2,.6)),'out_neighbor_jaccard':float(rng.uniform(0,2)),'in_neighbor_jaccard':float(rng.uniform(0,2)),'char_cosine':float(rng.uniform(0,1)),'word_cosine':float(rng.uniform(0,1)),'position_rank':float(rng.uniform(0,.4))})

search_rows=[]
for cache in fold_cache:
    fold=cache['fold']; x_val,y_val,q_val,c_val=cache['val']
    soft_x=x_val[:,[FEATURES.index(x) for x in SOFT_FEATURES]]
    tie_x=x_val[:,[FEATURES.index(x) for x in TIE_FEATURES]]
    for cfg in soft_weights:
        score=soft_x@np.asarray([cfg[x] for x in SOFT_FEATURES],np.float32)
        m=evaluate_scores(y_val,q_val,c_val,score); search_rows.append({'fold':fold,'family':'soft','config_id':cfg['search_id'],**m})
    for cfg in tie_weights:
        score=tie_x@np.asarray([cfg[x] for x in TIE_FEATURES],np.float32)
        m=evaluate_scores(y_val,q_val,c_val,score); search_rows.append({'fold':fold,'family':'hardmin','config_id':cfg['search_id'],**m})

search_results=pd.DataFrame(search_rows)
search_summary=search_results.groupby(['family','config_id']).agg(mean_accuracy=('accuracy','mean'),std_accuracy=('accuracy','std'),mean_top3=('top3','mean'),mean_top5=('top5','mean')).reset_index().sort_values('mean_accuracy',ascending=False)
best_soft_id=int(search_summary[search_summary.family.eq('soft')].iloc[0].config_id)
best_hardmin_id=int(search_summary[search_summary.family.eq('hardmin')].iloc[0].config_id)
best_soft=soft_weights[best_soft_id]; best_hardmin=tie_weights[best_hardmin_id]
display(search_summary.groupby('family').head(8))
print('Best soft config:',best_soft_id,best_soft)
print('Best hard-min config:',best_hardmin_id,best_hardmin)

## 11. Nested stability dan baseline comparison

Untuk mengukur selection optimism, konfigurasi terbaik dipilih memakai dua fold lalu diuji pada fold ketiga. Jika konfigurasi yang sama terus terpilih dan held-out score stabil, bobot lebih layak dibawa ke test dibanding konfigurasi yang hanya menang pada satu fold.

In [ ]:
def nested_selection(family):
    family_df=search_results[search_results.family.eq(family)]
    rows=[]
    for held_fold in range(N_SPLITS):
        fit_scores=family_df[family_df.fold.ne(held_fold)].groupby('config_id').accuracy.mean().sort_values(ascending=False)
        selected=int(fit_scores.index[0])
        held_score=float(family_df[(family_df.fold.eq(held_fold))&(family_df.config_id.eq(selected))].accuracy.iloc[0])
        rows.append({'held_fold':held_fold,'selected_config':selected,'selection_score':fit_scores.iloc[0],'held_accuracy':held_score})
    return pd.DataFrame(rows)

nested_soft=nested_selection('soft'); nested_hard=nested_selection('hardmin')
display(nested_soft); display(nested_hard)
print('Nested soft mean:',nested_soft.held_accuracy.mean())
print('Nested hard-min mean:',nested_hard.held_accuracy.mean())

v1_rows=[]; selected_rows=[]; oof_winners=[]
for cache in fold_cache:
    fold=cache['fold']; x_val,y_val,q_val,c_val=cache['val']; f={name:x_val[:,i] for i,name in enumerate(FEATURES)}
    v1_score=3*f['direct_target']+1.1*f['word_cosine']+1.6*f['char_cosine']+.9*f['category_exact_jaccard']-.20*f['graph_distance']+.45*f['graph_improvement']+.18*f['log_current_clicks']+.05*f['log_global_clicks']+.08*f['title_match_max']-.03*f['first_position']
    soft_score=x_val[:,[FEATURES.index(x) for x in SOFT_FEATURES]]@np.asarray([best_soft[x] for x in SOFT_FEATURES],np.float32)
    hard_score=x_val[:,[FEATURES.index(x) for x in TIE_FEATURES]]@np.asarray([best_hardmin[x] for x in TIE_FEATURES],np.float32)
    for name,score in [('v1_rebuilt',v1_score),('v3_soft_tuned',soft_score),('v3_recovered_hardmin',hard_score)]:
        m=evaluate_scores(y_val,q_val,c_val,score); selected_rows.append({'fold':fold,'model':name,**m})
    wt=winner_table(y_val,q_val,c_val,hard_score); wt['fold']=fold; oof_winners.append(wt)

selected_cv=pd.DataFrame(selected_rows)
display(selected_cv.pivot(index='model',columns='fold',values='accuracy'))
display(selected_cv.groupby('model').agg(mean_accuracy=('accuracy','mean'),std_accuracy=('accuracy','std'),mean_top3=('top3','mean'),mean_top5=('top5','mean'),candidate_recall=('candidate_recall','mean')).sort_values('mean_accuracy',ascending=False))

## 12. Leakage-safe LambdaRank sebagai learned comparator

V1 memakai binary candidate classifier. V2 menguji LambdaRank hanya pada kandidat minimum-distance dan menghapus query training yang truth-nya tidak berada pada subset tersebut. Tujuannya bukan memaksa learned model menjadi final, tetapi menguji apakah nonlinear tie-break benar-benar mengalahkan transparent tuned rule.

In [ ]:
ranker_rows=[]; fold_rankers=[]
if RUN_LAMBDARANK:
    try:
        import lightgbm as lgb
        TIE_MODEL_FEATURES=['category_exact_jaccard','category_overlap_count','category_prefix_jaccard','log_global_clicks','log_current_clicks','log_in_degree','log_out_degree','out_neighbor_jaccard','in_neighbor_jaccard','reciprocal_link','char_cosine','word_cosine','char_rank','word_rank','position_rank','ocr_conf_max','title_match_max']
        feature_idx=[FEATURES.index(x) for x in TIE_MODEL_FEATURES]; min_idx=FEATURES.index('is_min_neighbor_distance')
        for cache in fold_cache:
            fold=cache['fold']; x_fit,y_fit,q_fit,c_fit=cache['fit']; x_val,y_val,q_val,c_val=cache['val']
            fit_order=np.argsort(q_fit,kind='stable'); val_order=np.argsort(q_val,kind='stable')
            x_fit,y_fit,q_fit,c_fit=x_fit[fit_order],y_fit[fit_order],q_fit[fit_order],c_fit[fit_order]
            x_val,y_val,q_val,c_val=x_val[val_order],y_val[val_order],q_val[val_order],c_val[val_order]
            fit_min=x_fit[:,min_idx].eq(1) if isinstance(x_fit[:,min_idx],pd.Series) else x_fit[:,min_idx]==1
            val_min=x_val[:,min_idx]==1
            good_queries=set(q_fit[fit_min&(y_fit==1)].tolist()); fit_min &= np.isin(q_fit,list(good_queries))
            xt,yt,qt=x_fit[fit_min][:,feature_idx],y_fit[fit_min],q_fit[fit_min]
            xv,yv,qv,cv=x_val[val_min][:,feature_idx],y_val[val_min],q_val[val_min],c_val[val_min]
            train_groups=pd.Series(qt).value_counts(sort=False).sort_index().to_numpy(); val_groups=pd.Series(qv).value_counts(sort=False).sort_index().to_numpy()
            model=lgb.LGBMRanker(objective='lambdarank',metric='ndcg',n_estimators=350,learning_rate=.03,num_leaves=15,min_child_samples=30,colsample_bytree=.9,reg_lambda=3,random_state=SEED+fold,verbosity=-1)
            model.fit(xt,yt,group=train_groups,eval_set=[(xv,yv)],eval_group=[val_groups],callbacks=[lgb.early_stopping(50,verbose=False)])
            pred=model.predict(xv,num_iteration=model.best_iteration_); m=evaluate_scores(yv,qv,cv,pred)
            ranker_rows.append({'fold':fold,'model':'tie_lambdarank',**m,'best_iteration':model.best_iteration_}); fold_rankers.append(model)
        ranker_cv=pd.DataFrame(ranker_rows); display(ranker_cv); print('Tie LambdaRank mean:',ranker_cv.accuracy.mean())
    except Exception as exc:
        RUN_LAMBDARANK=False; ranker_cv=pd.DataFrame(); print('LambdaRank skipped:',type(exc).__name__,exc)
else:
    ranker_cv=pd.DataFrame(); print('LambdaRank disabled by configuration.')

## 13. Visual model comparison dan weight interpretation

Model dipilih berdasarkan mean grouped OOF top-1, dengan fold stability sebagai guardrail. Plot bobot menjelaskan tie-break akhir. Hard-min margin adalah constraint, bukan feature importance biasa; bobot setelahnya menentukan pilihan ketika beberapa kandidat sama-sama shortest-path.

In [ ]:
model_summary=selected_cv.groupby('model').agg(mean_accuracy=('accuracy','mean'),std_accuracy=('accuracy','std'),top3=('top3','mean'),top5=('top5','mean')).reset_index()
if len(ranker_cv):
    model_summary=pd.concat([model_summary,pd.DataFrame([{'model':'tie_lambdarank','mean_accuracy':ranker_cv.accuracy.mean(),'std_accuracy':ranker_cv.accuracy.std(),'top3':ranker_cv.top3.mean(),'top5':ranker_cv.top5.mean()}])],ignore_index=True)

fig,ax=plt.subplots(1,2,figsize=(16,5))
sns.barplot(data=model_summary.sort_values('mean_accuracy'),x='mean_accuracy',y='model',ax=ax[0],color='#4c78a8'); ax[0].set_xlim(0,1); ax[0].set_title('Grouped OOF top-1')
weight_plot=pd.Series({k:v for k,v in best_hardmin.items() if k not in ['search_id','is_min_neighbor_distance']}).sort_values()
weight_plot.plot.barh(ax=ax[1],color=np.where(weight_plot>=0,'#54a24b','#e45756')); ax[1].set_title('Hard-min tie-break weights')
plt.tight_layout(); plt.show()

fold_plot=selected_cv[selected_cv.model.ne('v1_rebuilt')]
fig,ax=plt.subplots(figsize=(9,4)); sns.barplot(data=fold_plot,x='fold',y='accuracy',hue='model',ax=ax); ax.set_ylim(0,1); ax.set_title('Fold stability'); plt.tight_layout(); plt.show()
display(model_summary.sort_values('mean_accuracy',ascending=False))

## 14. Error analysis berdasarkan jumlah shortest-path ties

Jika minimum-distance candidate unik, routing hampir deterministic. Ketika jumlah tie meningkat, semantic/category tie-break menjadi makin penting. Cell ini menggabungkan OOF prediction dengan tie count untuk menunjukkan regime yang masih menyisakan peluang V3.

In [ ]:
oof=pd.concat(oof_winners,ignore_index=True)
tie_frames=[]
for cache in fold_cache:
    x_val,_,q_val,_=cache['val']
    tie_frames.append(pd.DataFrame({'state_id':q_val,'tie_count':x_val[:,FEATURES.index('distance_tie_count')]}).drop_duplicates('state_id'))
tie_by_state=pd.concat(tie_frames).drop_duplicates('state_id')
oof=oof.merge(tie_by_state,on='state_id',how='left').merge(train[['state_id','current_article_id','target_article_id','next_article_id']],on='state_id',how='left')
oof['tie_bin']=pd.cut(oof.tie_count,bins=[0,1,2,3,5,10,np.inf],labels=['1','2','3','4-5','6-10','11+'])
error_slice=oof.groupby('tie_bin',observed=True).agg(states=('state_id','size'),accuracy=('correct','mean')).reset_index()
display(error_slice)
fig,ax=plt.subplots(1,2,figsize=(13,4)); sns.barplot(data=error_slice,x='tie_bin',y='accuracy',ax=ax[0],color='#72b7b2'); ax[0].set_ylim(0,1); ax[0].set_title('OOF accuracy by shortest-path tie count')
sns.barplot(data=error_slice,x='tie_bin',y='states',ax=ax[1],color='#f2cf5b'); ax[1].set_title('States per tie regime'); plt.tight_layout(); plt.show()
print('INSIGHT ERROR: fokus V3 sebaiknya pada tie regime dengan volume besar dan accuracy rendah, bukan pada unique-min states.')

## Error taxonomy V3

Error dipisahkan menjadi candidate missing, truth tidak berada pada shortest-path graph V3, dan shortest-path tie-break error. Breakdown ini mencegah kita salah mengira seluruh error sebagai kegagalan model ranking.

In [ ]:

taxonomy_rows=[]
for cache in fold_cache:
    fold=cache['fold']; x_val,y_val,q_val,c_val=cache['val']
    hard_score=x_val[:,[FEATURES.index(x) for x in TIE_FEATURES]]@np.asarray([best_hardmin[x] for x in TIE_FEATURES],np.float32)
    winners=winner_table(y_val,q_val,c_val,hard_score).set_index('state_id')
    for state_id in np.unique(q_val):
        mask=q_val==state_id; positive=mask&(y_val==1); present=bool(positive.any())
        truth_min=bool(present and x_val[positive,FEATURES.index('is_min_neighbor_distance')][0]==1)
        correct=bool(winners.loc[state_id,'correct'])
        kind='correct' if correct else ('candidate_missing' if not present else ('truth_not_shortest' if not truth_min else 'shortest_tiebreak_error'))
        taxonomy_rows.append({'fold':fold,'state_id':int(state_id),'error_type':kind,'tie_count':int(x_val[mask,FEATURES.index('distance_tie_count')][0])})
error_taxonomy=pd.DataFrame(taxonomy_rows)
taxonomy_summary=error_taxonomy.error_type.value_counts().rename_axis('error_type').reset_index(name='states'); taxonomy_summary['rate']=taxonomy_summary.states/len(error_taxonomy)
display(taxonomy_summary)
fig,ax=plt.subplots(figsize=(9,4)); sns.barplot(data=taxonomy_summary,x='error_type',y='states',ax=ax,color='#e45756'); ax.tick_params(axis='x',rotation=15); ax.set_title('V3 OOF outcome taxonomy'); plt.tight_layout(); plt.show()
print('INSIGHT TAXONOMY: candidate/graph recovery gain is now measurable separately from remaining tie-break errors.')


## 15. Final test inference

Tidak ada model fitting yang diperlukan untuk primary hard-min heuristic. Feature test memakai seluruh train sebagai fold-safe prior terhadap test. Notebook juga membuat soft-tuned alternative agar leaderboard dapat menguji apakah hard shortest-path constraint sesuai test distribution.

In [ ]:
full_priors=fit_priors(train)
x_test,_,q_test,c_test=build_pairs(test,full_priors,training=False)
hard_test=x_test[:,[FEATURES.index(x) for x in TIE_FEATURES]]@np.asarray([best_hardmin[x] for x in TIE_FEATURES],np.float32)
soft_test=x_test[:,[FEATURES.index(x) for x in SOFT_FEATURES]]@np.asarray([best_soft[x] for x in SOFT_FEATURES],np.float32)

hard_winners=winner_table(np.zeros(len(q_test),np.int8),q_test,c_test,hard_test)
soft_winners=winner_table(np.zeros(len(q_test),np.int8),q_test,c_test,soft_test)
hard_prediction=hard_winners.set_index('state_id').prediction.astype(int)
soft_prediction=soft_winners.set_index('state_id').prediction.astype(int)

primary=sample[['state_id']].copy(); primary['predicted_next_article_id']=primary.state_id.map(hard_prediction)
alternative=sample[['state_id']].copy(); alternative['predicted_next_article_id']=alternative.state_id.map(soft_prediction)
assert primary.predicted_next_article_id.notna().all() and alternative.predicted_next_article_id.notna().all()
primary['predicted_next_article_id']=primary.predicted_next_article_id.astype(int)
alternative['predicted_next_article_id']=alternative.predicted_next_article_id.astype(int)
print('Primary/alternative disagreement:',(primary.predicted_next_article_id!=alternative.predicted_next_article_id).mean())
display(primary.head())

## 16. Submission audit dan artifact export

Primary submission adalah hard-min tie-break dengan grouped OOF tertinggi. CSV alternatif tidak menggantikan primary; ia disediakan untuk controlled leaderboard comparison. Diagnostics, OOF, search table, weights, dan predictions disimpan agar V3 tidak perlu mengulang analisis dari nol.

In [ ]:
valid_ids=set(articles.article_id.astype(int))
def audit_submission(frame):
    return {
        'rows':len(frame),'unique_states':frame.state_id.nunique(),'nan':int(frame.predicted_next_article_id.isna().sum()),
        'invalid_article_ids':int((~frame.predicted_next_article_id.isin(valid_ids)).sum()),
        'same_as_current':float((frame.predicted_next_article_id.to_numpy()==test.current_article_id.to_numpy()).mean()),
        'same_as_target':float((frame.predicted_next_article_id.to_numpy()==test.target_article_id.to_numpy()).mean()),
        'unique_predictions':frame.predicted_next_article_id.nunique(),
    }

primary_audit=audit_submission(primary); alternative_audit=audit_submission(alternative)
assert primary_audit['rows']==6000 and primary_audit['unique_states']==6000
assert primary_audit['nan']==0 and primary_audit['invalid_article_ids']==0

primary_path=OUTPUT_DIR/'datathon_task2_v3_ocr_recovered_hardmin.csv'
alternative_path=OUTPUT_DIR/'datathon_task2_v3_soft_tuned.csv'
primary.to_csv(primary_path,index=False); alternative.to_csv(alternative_path,index=False)
selected_cv.to_csv(OUTPUT_DIR/'datathon_task2_v3_cv_selected_models.csv',index=False)
search_results.to_csv(OUTPUT_DIR/'datathon_task2_v3_weight_search.csv',index=False)
oof.to_csv(OUTPUT_DIR/'datathon_task2_v3_oof_hardmin.csv',index=False)
recovery.to_csv(OUTPUT_DIR/'datathon_task2_v3_recovered_edges.csv',index=False)
error_taxonomy.to_csv(OUTPUT_DIR/'datathon_task2_v3_error_taxonomy.csv',index=False)
(OUTPUT_DIR/'datathon_task2_v3_best_weights.json').write_text(json.dumps({'soft':best_soft,'hardmin':best_hardmin},indent=2))

diagnostics={
    'version':'v3_ocr_recovery_reverse_engineering','seed':SEED,'recovered_edges':len(recovery),'suspect_regions':len(suspect_regions),'seed':SEED,'reuse_path':str(REUSE_PATH),
    'runtime_minutes':(time.time()-START_TIME)/60,'route_diagnostics':route_df.mean(numeric_only=True).to_dict(),
    'selected_cv':selected_cv.groupby('model').mean(numeric_only=True).to_dict('index'),
    'nested_soft_accuracy':nested_soft.held_accuracy.mean(),'nested_hardmin_accuracy':nested_hard.held_accuracy.mean(),
    'best_soft':best_soft,'best_hardmin':best_hardmin,'primary_audit':primary_audit,'alternative_audit':alternative_audit,
}
(OUTPUT_DIR/'datathon_task2_v3_diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
archive=shutil.make_archive(str(OUTPUT_DIR.parent/'datathon_task2_v3_artifacts'),'zip',OUTPUT_DIR)
display(pd.DataFrame({'primary':primary_audit,'alternative':alternative_audit}))
print('Primary submission:',primary_path)
print('Alternative:',alternative_path)
print('Artifact bundle:',archive)
print(f"Total runtime: {(time.time()-START_TIME)/60:.2f} minutes")

## Keputusan setelah run

Urutan membaca hasil V3:

1. periksa jumlah recovered edges dan kenaikan candidate recall;
2. bandingkan V2 anchor 75,93% dengan V3 recovered hard-min;
3. baca error taxonomy untuk memisahkan extraction/graph/tie-break failures;
4. submit `datathon_task2_v3_ocr_recovered_hardmin.csv` jika grouped OOF meningkat stabil;
5. simpan executed notebook dan seluruh output untuk diagnosis V4.

Preliminary local evidence sebelum source V3 dibuat: OCR recovery menaikkan candidate recall menjadi sekitar 99,39% dan grouped OOF menjadi sekitar 76,90%. Leaderboard tetap menjadi external validation.